> **Not used for DeepSea 5×5.**  
> DeepSea observations are 25-dim one-hot vectors — already the most compact lossless representation possible. No autoencoder or representation learning is needed.  
> `embed_dim = obs_dim = 25` (identity mapping).  
> See `wiki/psdrl-autoencoder-analysis.md` for a full analysis of why PSDRL used a CNN autoencoder for Atari and why it is removed here.  
>  
> This notebook is provided for **future vision-based environments** where raw observations are high-dimensional images and a representation network is required before the GRU transition model.

# PSDRL Representation Network Hyperparameter Search

Tunes an **MLP autoencoder** for DeepSea 5×5 deterministic using Optuna NSGA-II multi-objective search.

> **Note:** The original PSDRL representation network (`PSDRL/networks/representation.py`) is a CNN autoencoder designed for 64×64 grayscale Atari frames. It is **not applicable** to DeepSea which uses vectorial 25-dim binary observations. This notebook tunes a replacement MLP autoencoder that compresses 25-dim observations into a latent embedding.

**Two objectives:**
- **Obj 1 — `reconstruction_loss`**: MSE between input and reconstructed observation on the test set. Lower = better reconstruction quality.
- **Obj 2 — `env_steps_to_accuracy`**: Minimum observations to achieve `reconstruction_loss < TARGET_RECON_LOSS`. Probed at 250, 500; falls back to 1000.

**Search space:** `embed_dim`, `hidden_dim`, `learning_rate`, `training_iterations`

The optimal `embed_dim` from this notebook determines the latent state size used by the transition and value networks in the full PSDRL agent.

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import json
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import NSGAIISampler
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from bsuite.environments import deep_sea

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Constants and Search Space

| Parameter | Choices | Description |
|---|---|---|
| `embed_dim` | [4, 8, 16, 25] | Latent bottleneck dimension (25 = lossless) |
| `hidden_dim` | [32, 64, 128] | Encoder/decoder hidden layer width |
| `learning_rate` | [1e-4, 1e-3, 3e-3, 1e-2] | Adam learning rate |
| `training_iterations` | [50, 100, 200, 500] | Gradient steps per training call |

`embed_dim=25` means no compression — the encoder is an identity-like mapping. Smaller values force more aggressive compression.

In [ ]:
# --- Environment ---
DS_SIZE     = 5
N_ACTIONS   = 2
OBS_DIM     = DS_SIZE * DS_SIZE   # 25

# --- Search space ---
EMBED_DIM_CHOICES = [4, 8, 16, 25]
HIDDEN_DIM_CHOICES = [32, 64, 128]
LR_CHOICES         = [1e-4, 1e-3, 3e-3, 1e-2]
ITER_CHOICES       = [50, 100, 200, 500]

# --- Data ---
TIME_LIMIT        = 1000
TEST_SIZE         = 200
PROBE_SIZES       = [250, 500]
TARGET_RECON_LOSS = 0.01  # MSE threshold — close to perfect for binary obs

# --- Training ---
BATCH_SIZE    = 64
PATIENCE      = 30

# --- Study ---
N_TRIALS      = 60

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## MLP AutoEncoder

Replacement for the CNN autoencoder. Compresses 25-dim binary observations through a linear bottleneck.

Architecture: `25 → hidden_dim → embed_dim → hidden_dim → 25`

Sigmoid output: DeepSea observations are binary (one-hot in [0, 1]).

In [ ]:
class MLPAutoEncoder(nn.Module):
    def __init__(self, obs_dim, embed_dim, hidden_dim, lr, device):
        super().__init__()
        self.embed_dim = embed_dim
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, obs_dim),
            nn.Sigmoid(),
        )
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        self.loss_fn   = nn.MSELoss()
        self.to(device)

    def embed(self, obs):
        with torch.no_grad():
            return self.encoder(obs)

    def forward(self, obs):
        return self.decoder(self.encoder(obs))

    def reconstruction_loss(self, obs):
        return self.loss_fn(self.forward(obs), obs)

## Data Collection

In [ ]:
def collect_observations(n_total, ds_size=5, seed=42):
    """Collect n_total flattened DeepSea observations."""
    env = deep_sea.DeepSea(size=ds_size, deterministic=True, seed=seed)
    obs_list = []
    total = 0
    while total < n_total:
        ts = env.reset()
        while True:
            obs = torch.FloatTensor(ts.observation.flatten())
            obs_list.append(obs)
            total += 1
            action = np.random.randint(0, N_ACTIONS)
            ts = env.step(action)
            if ts.last():
                break
    return torch.stack(obs_list[:n_total])


all_obs = collect_observations(TIME_LIMIT + TEST_SIZE)
train_obs = all_obs[:TIME_LIMIT]
test_obs  = all_obs[TIME_LIMIT:TIME_LIMIT + TEST_SIZE]

print(f'Train obs: {train_obs.shape}, Test obs: {test_obs.shape}')
print(f'Obs range: [{train_obs.min():.2f}, {train_obs.max():.2f}]')

## Training and Evaluation Functions

In [ ]:
def train_ae(ae, obs_data, n_iters, batch_size=BATCH_SIZE,
             patience=PATIENCE, device=DEVICE):
    N = len(obs_data)
    obs_t = obs_data.to(device)
    best_loss = float('inf')
    patience_ctr = 0
    epochs_to_converge = n_iters

    for iteration in range(n_iters):
        idx  = torch.randint(0, N, (batch_size,))
        batch = obs_t[idx]
        ae.optimizer.zero_grad()
        loss = ae.reconstruction_loss(batch)
        loss.backward()
        ae.optimizer.step()

        if loss.item() < best_loss - 1e-6:
            best_loss = loss.item()
            patience_ctr = 0
            epochs_to_converge = iteration + 1
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    return best_loss, epochs_to_converge


def eval_ae(ae, obs_data, device=DEVICE):
    """MSE reconstruction loss on obs_data."""
    ae.eval()
    with torch.no_grad():
        loss = ae.reconstruction_loss(obs_data.to(device)).item()
    ae.train()
    return loss

## Optuna Objective

In [ ]:
def objective(trial):
    embed_dim  = trial.suggest_categorical('embed_dim',            EMBED_DIM_CHOICES)
    hidden_dim = trial.suggest_categorical('hidden_dim',           HIDDEN_DIM_CHOICES)
    lr         = trial.suggest_categorical('learning_rate',        LR_CHOICES)
    n_iters    = trial.suggest_categorical('training_iterations',  ITER_CHOICES)

    # --- Objective 1: reconstruction loss on full dataset ---
    ae = MLPAutoEncoder(OBS_DIM, embed_dim, hidden_dim, lr, DEVICE)
    _, epochs_conv = train_ae(ae, train_obs, n_iters=n_iters, device=DEVICE)
    recon_loss = eval_ae(ae, test_obs, device=DEVICE)
    trial.set_user_attr('epochs_to_converge', epochs_conv)

    # --- Objective 2: env_steps_to_accuracy (sample efficiency) ---
    env_steps_to_accuracy = TIME_LIMIT
    for n in PROBE_SIZES:
        probe_ae = MLPAutoEncoder(OBS_DIM, embed_dim, hidden_dim, lr, DEVICE)
        train_ae(probe_ae, train_obs[:n], n_iters=n_iters // 2, device=DEVICE)
        probe_loss = eval_ae(probe_ae, test_obs, device=DEVICE)
        if probe_loss < TARGET_RECON_LOSS:
            env_steps_to_accuracy = n
            break

    return recon_loss, float(env_steps_to_accuracy)

In [ ]:
sampler = NSGAIISampler(seed=42)
study = optuna.create_study(
    directions=['minimize', 'minimize'],
    sampler=sampler,
    study_name='psdrl_representation',
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\nCompleted {len(study.trials)} trials')

## Results Analysis

In [ ]:
def trials_to_df(study):
    rows = []
    for t in study.trials:
        if t.values is None:
            continue
        row = dict(t.params)
        row['reconstruction_loss']    = t.values[0]
        row['env_steps_to_accuracy']  = t.values[1]
        row['epochs_to_converge']     = t.user_attrs.get('epochs_to_converge', 500)
        row['pareto']                 = t in study.best_trials
        rows.append(row)
    return pd.DataFrame(rows)


df = trials_to_df(study)
pareto_df = df[df['pareto']].sort_values('reconstruction_loss')

print(f'Total trials: {len(df)}')
print(f'Pareto-front trials: {len(pareto_df)}')
pareto_df[['embed_dim','hidden_dim','learning_rate','training_iterations',
           'reconstruction_loss','env_steps_to_accuracy','epochs_to_converge']].round(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(df['env_steps_to_accuracy'], df['reconstruction_loss'],
           c='lightsteelblue', alpha=0.5, s=30, label='All trials')
ax.scatter(pareto_df['env_steps_to_accuracy'], pareto_df['reconstruction_loss'],
           c='crimson', s=80, zorder=5, label='Pareto front')
ax.axhline(TARGET_RECON_LOSS, color='orange', linestyle='--',
           alpha=0.7, label=f'Target ({TARGET_RECON_LOSS})')
ax.set_xlabel('Env Steps to Accuracy (Obj 2)')
ax.set_ylabel('Reconstruction Loss (Obj 1)')
ax.set_title('PSDRL Representation — Pareto Front')
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
colors = {4: 'red', 8: 'orange', 16: 'green', 25: 'blue'}
for ed in sorted(df['embed_dim'].unique()):
    sub = df[df['embed_dim'] == ed]
    ax2.scatter(sub['env_steps_to_accuracy'], sub['reconstruction_loss'],
                alpha=0.6, s=40, label=f'embed_dim={ed}',
                color=colors.get(ed, 'gray'))
ax2.set_xlabel('Env Steps to Accuracy')
ax2.set_ylabel('Reconstruction Loss')
ax2.set_title('Coloured by embed_dim')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Best Model Evaluation

In [ ]:
if len(pareto_df) > 0:
    pdf = pareto_df.copy()
    for col in ['reconstruction_loss', 'env_steps_to_accuracy']:
        rng = pdf[col].max() - pdf[col].min()
        pdf[col + '_norm'] = (pdf[col] - pdf[col].min()) / (rng if rng > 0 else 1)
    pdf['score'] = pdf['reconstruction_loss_norm'] + pdf['env_steps_to_accuracy_norm']
    best_row = pdf.loc[pdf['score'].idxmin()]
else:
    best_row = df.loc[df['reconstruction_loss'].idxmin()]

best_embed  = int(best_row['embed_dim'])
best_hidden = int(best_row['hidden_dim'])
best_lr     = float(best_row['learning_rate'])
best_iters  = int(best_row['training_iterations'])

print('Best config:', best_row[['embed_dim','hidden_dim','learning_rate','training_iterations']].to_dict())
print(f'reconstruction_loss={best_row["reconstruction_loss"]:.6f}')
print(f'env_steps_to_accuracy={best_row["env_steps_to_accuracy"]:.0f}')

best_ae = MLPAutoEncoder(OBS_DIM, best_embed, best_hidden, best_lr, DEVICE)
final_loss, final_epochs = train_ae(
    best_ae, train_obs, n_iters=best_iters, device=DEVICE
)
final_recon = eval_ae(best_ae, test_obs, device=DEVICE)
print(f'Retrained best model — train_loss: {final_loss:.6f}, test_loss: {final_recon:.6f}')

In [ ]:
# Visualise reconstruction quality: input vs reconstructed for a few test samples
N_GRID = DS_SIZE
best_ae.eval()

with torch.no_grad():
    sample_obs = test_obs[:8].to(DEVICE)
    recons     = best_ae.forward(sample_obs).cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Top: Input observations | Bottom: Reconstructions', y=1.02)
for i in range(8):
    axes[0, i].imshow(sample_obs[i].cpu().numpy().reshape(N_GRID, N_GRID),
                      cmap='Blues', vmin=0, vmax=1)
    axes[0, i].axis('off')
    state = sample_obs[i].cpu().argmax().item()
    axes[0, i].set_title(f's={state}', fontsize=7)

    axes[1, i].imshow(recons[i].reshape(N_GRID, N_GRID),
                      cmap='Blues', vmin=0, vmax=1)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

best_ae.train()

# Embedding space visualisation (only if embed_dim <= 2 for scatter)
if best_embed <= 2:
    with torch.no_grad():
        embeds = best_ae.embed(test_obs.to(DEVICE)).cpu().numpy()
    states = test_obs.argmax(dim=-1).numpy()
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(embeds[:, 0], embeds[:, 1] if best_embed > 1 else np.zeros(len(embeds)),
                    c=states, cmap='tab20', s=40, alpha=0.7)
    plt.colorbar(sc, ax=ax, label='State index')
    ax.set_xlabel('Embedding dim 0')
    ax.set_ylabel('Embedding dim 1' if best_embed > 1 else '(1D projection)')
    ax.set_title(f'Latent Space (embed_dim={best_embed})')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'embed_dim={best_embed} — skipping 2D scatter (too many dimensions)')

## Save Optimal Parameters

In [ ]:
optimal_params = {
    'embed_dim':             int(best_row['embed_dim']),
    'hidden_dim':            int(best_row['hidden_dim']),
    'learning_rate':         float(best_row['learning_rate']),
    'training_iterations':   int(best_row['training_iterations']),
    'reconstruction_loss':   float(best_row['reconstruction_loss']),
    'env_steps_to_accuracy': int(best_row['env_steps_to_accuracy']),
    'epochs_to_converge':    int(best_row['epochs_to_converge']),
    'final_loss':            float(final_loss),
    'final_recon_loss':      float(final_recon),
    'obs_dim':               OBS_DIM,
    'n_actions':             N_ACTIONS,
    'ds_size':               DS_SIZE,
    'n_trials':              N_TRIALS,
    'data_samples':          TIME_LIMIT,
    'note': 'embed_dim from this notebook is the latent state size for the '
            'full PSDRL agent. Update transition and value notebooks to use '
            'this embed_dim instead of OBS_DIM as the raw input dimension.'
}

out_path = Path('configs/psdrl_representation_params.json')
out_path.parent.mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(optimal_params, f, indent=2)
print(f'Saved to {out_path}')
print(json.dumps(optimal_params, indent=2))